In [ ]:
BASE_DIR = "/content/drive/MyDrive/sports_video_summarizer"

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

os.makedirs(BASE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get install tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
!pip install pytesseract qdrant_client pydub ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.1 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import librosa
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoFeatureExtractor, AutoModel, AutoTokenizer, BertModel
from transformers import pipeline
import datetime
from qdrant_client import QdrantClient
from qdrant_client.http import models
import uuid
from ultralytics import YOLO
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import moviepy.editor as mp
from moviepy.editor import VideoFileClip
from pydub import AudioSegment
import json
import glob
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import yaml
import random
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)

  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)

  elif re.match('(flt)p?( \(default\))?$', token):

  elif re.m

In [ ]:
class AttentionMemoryNetwork(nn.Module):
    """Neural network for attention-based memory retrieval."""

    def __init__(self, input_dim, hidden_dim):
        super(AttentionMemoryNetwork, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.query_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.key_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.value_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.output_layer = nn.Linear(hidden_dim, 1)

    def forward(self, query, keys, values):

        transformed_query = self.query_transform(query).unsqueeze(1)
        transformed_keys = self.key_transform(keys)
        transformed_values = self.value_transform(values)

        attention_scores = torch.bmm(transformed_query, transformed_keys.transpose(1, 2))
        attention_scores = attention_scores / torch.sqrt(torch.tensor(self.hidden_dim, dtype=torch.float32))
        attention_weights = F.softmax(attention_scores, dim=2)

        context = torch.bmm(attention_weights, transformed_values)
        context = context.squeeze(1)

        output = self.output_layer(context)

        return output, attention_weights.squeeze(1)


In [ ]:
class AttentionDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        query = torch.tensor(item["query_embedding"], dtype=torch.float32)
        keys = torch.tensor(item["frame_features"], dtype=torch.float32)
        values = torch.tensor(item["text_features"], dtype=torch.float32)
        targets = torch.tensor(item["relevance_scores"], dtype=torch.float32)

        return query, keys, values, targets

In [ ]:
class SportsVideoSummarizer:
    def __init__(self, video_path, output_dir=os.path.join(BASE_DIR, "output"), frame_rate=1):
        self.video_path = video_path
        self.output_dir = output_dir

        self.frame_rate = frame_rate

        self.frames_dir = os.path.join(self.output_dir, "frames")
        self.summary_dir = os.path.join(self.output_dir, "summaries")
        self.model_dir = os.path.join(self.output_dir, "models")

        os.makedirs(self.frames_dir, exist_ok=True)
        os.makedirs(self.summary_dir, exist_ok=True)
        os.makedirs(self.model_dir, exist_ok=True)

        print("Loading models...")
        self.yolo_model = YOLO("yolov8n.pt")
        self.image_extractor = AutoFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
        self.image_model = AutoModel.from_pretrained("google/vit-base-patch16-224")
        self.text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        self.text_model = BertModel.from_pretrained("bert-base-uncased")
        self.sentiment_analyzer = pipeline("sentiment-analysis")
        self.db_client = QdrantClient(":memory:")
        self.collection_name = "sports_video_features"
        self.setup_vector_db()
        try:
            nltk.data.find('tokenizers/punkt')
            nltk.data.find('corpora/stopwords')
        except LookupError:
            nltk.download('punkt')
            nltk.download('punkt_tab')
            nltk.download('stopwords')
        self.stop_words = set(stopwords.words('english'))
        self.attention_weights = {"visual": 0.4, "audio": 0.3, "text": 0.3}
        self.attention_network = AttentionMemoryNetwork(768, 256)
        self.excitement_thresholds = {"rms_mean": 0.05, "rms_max": 0.1, "tempo": 120, "contrast_mean": 15}
        print("Initialization complete.")

    def setup_vector_db(self):
        visual_dim = 768
        text_dim = 768
        self.db_client.recreate_collection(
            collection_name=self.collection_name,
            vectors_config={
                "visual": models.VectorParams(size=visual_dim, distance=models.Distance.COSINE),
                "text": models.VectorParams(size=text_dim, distance=models.Distance.COSINE)
            }
        )

    def preprocess_video(self):
        print(f"Preprocessing video: {self.video_path}")
        frames, timestamps = self.extract_frames()
        print(f"\n{len(frames)}\n")
        audio_path = self.extract_audio()
        frame_features = []
        for i, (frame, timestamp) in enumerate(zip(frames, timestamps)):
            print(f"Processing frame {i+1}/{len(frames)} at {timestamp:.2f}s")
            frame_data = self.process_frame(frame, timestamp, i)
            frame_features.append(frame_data)
            if (i + 1) % 100 == 0:
                self.save_progress(frame_features)
        self.save_progress(frame_features)
        scenes = self.segment_scenes(frame_features)
        return frames, timestamps, audio_path, frame_features, scenes

    def extract_frames(self):
        """Extract frames from the video based on frame rate using moviepy."""
        try:
          clip = VideoFileClip(self.video_path)
          fps = clip.fps
          duration = clip.duration
          total_frames = int(fps * duration)

          print(f"Video properties: FPS={fps:.2f}, Duration={duration:.2f}s, Estimated Frames={total_frames}")

          frame_interval = int(round(fps / self.frame_rate))
          frames = []
          timestamps = []

          target_times = [t for t in np.arange(0, duration, 1/self.frame_rate)]

          for i, t in enumerate(target_times):
              try:
                  frame = clip.get_frame(t)
                  frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
                  frames.append(frame_bgr)
                  timestamps.append(t)
                  frame_count = i * frame_interval
                  cv2.imwrite(f"{self.frames_dir}/frame_{frame_count:05d}.jpg", frame_bgr)

                  if (i + 1) % 100 == 0:
                      print(f"Extracted {i + 1} frames so far...")

              except Exception as e:
                  print(f"Error extracting frame at {t:.2f}s: {str(e)}")
                  continue

          print(f"Total frames extracted: {len(frames)}")
          return frames, timestamps

        except Exception as e:
            raise ValueError(f"Error processing video with moviepy: {str(e)}")
        finally:
            if 'clip' in locals():
                clip.close()


    def extract_audio(self):
        video_name = os.path.splitext(os.path.basename(self.video_path))[0]
        audio_path = os.path.join(self.output_dir, f"{video_name}_audio.wav")
        if os.path.exists(audio_path):
            print(f"Audio already extracted: {audio_path}")
            return audio_path
        print("Extracting audio from video...")
        video = mp.VideoFileClip(self.video_path)
        video.audio.write_audiofile(audio_path)
        return audio_path

    def process_frame(self, frame, timestamp, frame_id):
        visual_features = self.extract_visual_features(frame)
        objects = self.detect_objects(frame)
        text = self.extract_text(frame)
        frame_data = {
            "id": frame_id,
            "timestamp": timestamp,
            "timestamp_formatted": str(datetime.timedelta(seconds=timestamp)),
            "frame_path": f"{self.frames_dir}/frame_{frame_id:05d}.jpg",
            "visual_features": visual_features,
            "objects": objects,
            "text": text,
            "text_embedding": self.get_text_embedding(text) if text else np.zeros(768).tolist()
        }
        self.store_frame_features(frame_data)
        return frame_data

    def extract_visual_features(self, frame):
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        inputs = self.image_extractor(pil_image, return_tensors="pt")
        with torch.no_grad():
            outputs = self.image_model(**inputs)
        feature_vector = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
        return feature_vector.tolist()

    def detect_objects(self, frame):
        results = self.yolo_model(frame)
        objects = []
        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = box.conf[0].item()
                cls = int(box.cls[0].item())
                name = result.names[cls]
                if conf > 0.4:
                    objects.append({"name": name, "confidence": conf, "bbox": [x1, y1, x2, y2]})
        return objects

    def extract_text(self, frame):
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        text = pytesseract.image_to_string(frame_rgb)
        return text.strip()

    def get_text_embedding(self, text):
        if not text:
            return np.zeros(768).tolist()
        inputs = self.text_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = self.text_model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]. squeeze().numpy()
        return embedding.tolist()

    def store_frame_features(self, frame_data):
        self.db_client.upsert(
            collection_name=self.collection_name,
            points=[
                models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector={
                        "visual": frame_data["visual_features"],
                        "text": frame_data["text_embedding"]
                    },
                    payload={
                        "frame_id": frame_data["id"],
                        "timestamp": frame_data["timestamp"],
                        "timestamp_formatted": frame_data["timestamp_formatted"],
                        "frame_path": frame_data["frame_path"],
                        "objects": frame_data["objects"],
                        "text": frame_data["text"]
                    }
                )
            ]
        )

    def save_progress(self, frame_features):
        progress_path = os.path.join(self.output_dir, "processing_progress.json")
        with open(progress_path, 'w') as f:
            json.dump({
                "video_path": self.video_path,
                "frames_processed": len(frame_features),
                "last_timestamp": frame_features[-1]["timestamp"] if frame_features else 0
            }, f)

    def segment_scenes(self, frame_features):
        print("Segmenting video into scenes...")
        features = np.array([np.array(frame["visual_features"]) for frame in frame_features])
        video_duration = frame_features[-1]["timestamp"] if frame_features else 0
        num_clusters = max(5, int(video_duration / 60))
        kmeans = KMeans(n_clusters=num_clusters, random_state=42)
        clusters = kmeans.fit_predict(features)
        scenes = []
        current_scene = {"cluster": clusters[0], "frames": [0]}
        for i in range(1, len(clusters)):
            if clusters[i] == current_scene["cluster"]:
                current_scene["frames"].append(i)
            else:
                start_time = frame_features[current_scene["frames"][0]]["timestamp"]
                end_time = frame_features[current_scene["frames"][-1]]["timestamp"]
                scenes.append({
                    "cluster": current_scene["cluster"],
                    "frame_indices": current_scene["frames"],
                    "start_time": start_time,
                    "end_time": end_time,
                    "duration": end_time - start_time
                })
                current_scene = {"cluster": clusters[i], "frames": [i]}
        if current_scene["frames"]:
            start_time = frame_features[current_scene["frames"][0]]["timestamp"]
            end_time = frame_features[current_scene["frames"][-1]]["timestamp"]
            scenes.append({
                "cluster": current_scene["cluster"],
                "frame_indices": current_scene["frames"],
                "start_time": start_time,
                "end_time": end_time,
                "duration": end_time - start_time
            })
        print(f"Video segmented into {len(scenes)} scenes")
        return scenes

    def analyze_audio_segment(self, audio_path, start_time, end_time):
        audio = AudioSegment.from_file(audio_path)
        start_ms = int(start_time * 1000)
        end_ms = int(end_time * 1000)
        segment = audio[start_ms:end_ms]
        temp_path = os.path.join(self.output_dir, "temp_segment.wav")
        segment.export(temp_path, format="wav")
        y, sr = librosa.load(temp_path)
        rms = librosa.feature.rms(y=y)[0]
        rms_mean = np.mean(rms)
        rms_std = np.std(rms)
        rms_max = np.max(rms)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = np.mean(contrast)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        os.remove(temp_path)
        excitement_score = (
            0.4 * (rms_max / self.excitement_thresholds["rms_max"]) +
            0.3 * (contrast_mean / self.excitement_thresholds["contrast_mean"]) +
            0.3 * (tempo / self.excitement_thresholds["tempo"])
        )
        excitement_score = min(max(excitement_score, 0), 1)
        return {
            "excitement_score": excitement_score,
            "rms_mean": rms_mean,
            "rms_max": rms_max,
            "tempo": tempo,
            "contrast_mean": contrast_mean
        }

    def process_query(self, query_text):
        print(f"Processing query: {query_text}")
        tokens = word_tokenize(query_text.lower())
        filtered_tokens = [word for word in tokens if word.isalnum() and word not in self.stop_words]
        keywords = filtered_tokens
        query_embedding = self.get_text_embedding(query_text)
        return {"original_query": query_text, "keywords": keywords, "embedding": query_embedding}

    def attention_mechanism(self, query, frame_features):
        print("Applying attention mechanism to score frames...")
        query_embedding = np.array(query["embedding"])
        if hasattr(self, 'attention_network_trained') and self.attention_network_trained:
            return self.attention_mechanism_neural(query, frame_features)
        scored_frames = []
        for frame in frame_features:
            visual_score = 0
            for obj in frame["objects"]:
                if obj["name"].lower() in query["keywords"]:
                    visual_score += obj["confidence"]
            if frame["objects"]:
                visual_score = min(visual_score, 1.0)
            text_score = 0
            if frame["text"]:
                frame_text_embedding = np.array(frame["text_embedding"])
                text_score = cosine_similarity([query_embedding], [frame_text_embedding])[0][0]
                text_score = (text_score + 1) / 2
            attention_score = (
                self.attention_weights["visual"] * visual_score +
                self.attention_weights["text"] * text_score
            )
            scored_frame = frame.copy()
            scored_frame["attention_score"] = attention_score
            scored_frames.append(scored_frame)
        sorted_frames = sorted(scored_frames, key=lambda x: x["attention_score"], reverse=True)
        return sorted_frames

    def attention_mechanism_neural(self, query, frame_features):
        print("Applying neural attention mechanism...")
        query_tensor = torch.tensor(query["embedding"], dtype=torch.float32)
        memory_keys = []
        memory_values = []
        for frame in frame_features:
            visual_features = torch.tensor(frame["visual_features"], dtype=torch.float32)
            memory_keys.append(visual_features)
            text_features = torch.tensor(frame["text_embedding"], dtype=torch.float32)
            memory_values.append(text_features)
        memory_keys = torch.stack(memory_keys)
        memory_values = torch.stack(memory_values)
        with torch.no_grad():
            _, attention_weights = self.attention_network(query_tensor, memory_keys, memory_values)
        attention_scores = attention_weights.squeeze().numpy()
        scored_frames = []
        for i, frame in enumerate(frame_features):
            scored_frame = frame.copy()
            scored_frame["attention_score"] = float(attention_scores[i])
            scored_frames.append(scored_frame)
        sorted_frames = sorted(scored_frames, key=lambda x: x["attention_score"], reverse=True)
        return sorted_frames

    def generate_summary(self, query, frames, timestamps, audio_path, frame_features, scenes):
        query_info = self.process_query(query)
        scored_frames = self.attention_mechanism(query_info, frame_features)
        top_frame_ids = [frame["id"] for frame in scored_frames[:int(len(scored_frames) * 0.2)]]
        scene_scores = {}
        for scene in scenes:
            matched_frames = set(scene["frame_indices"]) & set(top_frame_ids)
            if matched_frames:
                scene_scores[scene["cluster"]] = len(matched_frames) / len(scene["frame_indices"])
        selected_scenes = []
        for scene in scenes:
            if scene["cluster"] in scene_scores and scene_scores[scene["cluster"]] > 0.3:
                audio_analysis = self.analyze_audio_segment(audio_path, scene["start_time"], scene["end_time"])
                scene_with_score = scene.copy()
                scene_with_score["relevance_score"] = scene_scores[scene["cluster"]]
                scene_with_score["excitement_score"] = audio_analysis["excitement_score"]
                scene_with_score["combined_score"] = (
                    0.7 * scene_scores[scene["cluster"]] +
                    0.3 * audio_analysis["excitement_score"]
                )
                selected_scenes.append(scene_with_score)
        selected_scenes = sorted(selected_scenes, key=lambda x: x["combined_score"], reverse=True)
        max_duration = 300
        final_scenes = []
        total_duration = 0
        for scene in selected_scenes:
            if total_duration + scene["duration"] <= max_duration:
                final_scenes.append(scene)
                total_duration += scene["duration"]
            else:
                break
        final_scenes = sorted(final_scenes, key=lambda x: x["start_time"])
        print(f"Generated summary: {len(final_scenes)} scenes, {total_duration:.2f} seconds")
        summary_video_path = self.create_summary_video(final_scenes)
        return {
            "query": query,
            "summary_video_path": summary_video_path,
            "scenes": final_scenes,
            "duration": total_duration
        }

    def create_summary_video(self, scenes):
        if not scenes:
            print("No scenes selected for summary")
            return None
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(self.summary_dir, f"summary_{timestamp}.mp4")
        video = mp.VideoFileClip(self.video_path)
        clips = []
        for scene in scenes:
            subclip = video.subclip(scene["start_time"], scene["end_time"])
            clips.append(subclip)
        final_clip = mp.concatenate_videoclips(clips)
        final_clip.write_videofile(output_path)
        print(f"Summary video saved to {output_path}")
        return output_path

    def finetune_object_detection(self, dataset_path, epochs=100, batch_size=16):
        print(f"Fine-tuning YOLO model on sports dataset: {dataset_path}")
        config_path = os.path.join(self.model_dir, "sports_yolo_config.yaml")
        config = {
            "path": dataset_path,
            "train": os.path.join(dataset_path, "train/images"),
            "val": os.path.join(dataset_path, "valid/images"),
            "test": os.path.join(dataset_path, "test/images"),
            "names": {
                0: "player",
                1: "ball",
                2: "four",
                3: "six",
                4: "umpire",
                5: "scoreboard"
            }
        }
        with open(config_path, 'w') as f:
            yaml.dump(config, f)
        results = self.yolo_model.train(
            data=config_path,
            epochs=epochs,
            batch=batch_size,
            imgsz=640,
            save=True,
            project=self.model_dir,
            name="sports_yolo",
            exist_ok=True
        )
        best_weights_path = os.path.join(self.model_dir, "sports_yolo", "weights", "best.pt")
        if os.path.exists(best_weights_path):
            self.yolo_model = YOLO(best_weights_path)
            print(f"Loaded fine-tuned YOLO model from {best_weights_path}")
        else:
            print("Could not find fine-tuned model weights. Using original model.")
        return results

    def optimize_attention_weights(self, labeled_data_path):
        print(f"Optimizing attention weights using labeled data: {labeled_data_path}")
        with open(labeled_data_path, 'r') as f:
            labeled_data = json.load(f)
        best_score = 0
        best_weights = self.attention_weights.copy()
        visual_weights = [0.3, 0.4, 0.5, 0.6, 0.7]
        text_weights = [0.2, 0.3, 0.4, 0.5]
        for vw in visual_weights:
            for tw in text_weights:
                aw = 1.0 - vw - tw
                if aw < 0:
                    continue
                test_weights = {"visual": vw, "text": tw, "audio": aw}
                score = self._evaluate_attention_weights(test_weights, labeled_data)
                print(f"Weights: {test_weights}, Score: {score:.4f}")
                if score > best_score:
                    best_score = score
                    best_weights = test_weights
        print(f"Best weights found: {best_weights} with score {best_score:.4f}")
        self.attention_weights = best_weights
        weights_path = os.path.join(self.model_dir, "attention_weights.json")
        with open(weights_path, 'w') as f:
            json.dump(best_weights, f)
        return best_weights

    def _evaluate_attention_weights(self, weights, labeled_data):
        original_weights = self.attention_weights.copy()
        self.attention_weights = weights
        total_iou = 0
        total_examples = 0
        for example in labeled_data:
            if example["video_path"] != self.video_path:
                continue
            query = example["query"]
            ground_truth = example["ground_truth"]
            frames, timestamps, audio_path, frame_features, scenes = self.preprocess_video()
            summary = self.generate_summary(query, frames, timestamps, audio_path, frame_features, scenes)
            predicted_scenes = summary["scenes"]
            iou_score = self._calculate_temporal_iou(predicted_scenes, ground_truth)
            total_iou += iou_score
            total_examples += 1
        self.attention_weights = original_weights
        avg_score = total_iou / total_examples if total_examples > 0 else 0
        return avg_score

    def _calculate_temporal_iou(self, predicted_scenes, ground_truth):
        predicted_intervals = [(scene["start_time"], scene["end_time"]) for scene in predicted_scenes]
        ground_truth_intervals = [(scene["start_time"], scene["end_time"]) for scene in ground_truth]
        total_iou = 0
        for pred_interval in predicted_intervals:
            best_iou = 0
            for gt_interval in ground_truth_intervals:
                intersection_start = max(pred_interval[0], gt_interval[0])
                intersection_end = min(pred_interval[1], gt_interval[1])
                intersection = max(0, intersection_end - intersection_start)
                union_start = min(pred_interval[0], gt_interval[0])
                union_end = max(pred_interval[1], gt_interval[1])
                union = union_end - union_start
                iou = intersection / union if union > 0 else 0
                best_iou = max(best_iou, iou)
            total_iou += best_iou
        avg_iou = total_iou / len(predicted_intervals) if predicted_intervals else 0
        return avg_iou

    def train_attention_network(self, dataset_path, epochs=50, batch_size=16, learning_rate=0.001):
        print(f"Training attention network with dataset: {dataset_path}")
        with open(dataset_path, 'r') as f:
            dataset = json.load(f)
        train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=42)
        train_dataset = AttentionDataset(train_data)
        val_dataset = AttentionDataset(val_data)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
        input_dim = 768
        hidden_dim = 256
        self.attention_network = AttentionMemoryNetwork(input_dim, hidden_dim)
        criterion = nn.MSELoss()
        optimizer = Adam(self.attention_network.parameters(), lr=learning_rate)
        best_val_loss = float('inf')
        for epoch in range(epochs):
            self.attention_network.train()
            train_loss = 0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
                queries, keys, values, targets = batch
                outputs, _ = self.attention_network(queries, keys, values)
                loss = criterion(outputs, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            avg_train_loss = train_loss / len(train_loader)
            self.attention_network.eval()
            val_loss = 0
            with torch.no_grad():
                for batch in val_loader:
                    queries, keys, values, targets = batch
                    outputs, _ = self.attention_network(queries, keys, values)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                model_path = os.path.join(self.model_dir, "attention_network.pt")
                torch.save(self.attention_network.state_dict(), model_path)
                print(f"Model saved to {model_path}")
        self.attention_network.load_state_dict(torch.load(os.path.join(self.model_dir, "attention_network.pt")))
        self.attention_network_trained = True
        return {"train_loss": avg_train_loss, "val_loss": best_val_loss}

    def calibrate_excitement_detection(self, excitement_data_path):
        print(f"Calibrating excitement detection with data: {excitement_data_path}")
        with open(excitement_data_path, 'r') as f:
            excitement_data = json.load(f)
        audio_features = []
        excitement_labels = []
        for item in excitement_data:
            if os.path.exists(item["audio_path"]):
                y, sr = librosa.load(item["audio_path"])
                rms = librosa.feature.rms(y=y)[0]
                rms_mean = np.mean(rms)
                rms_max = np.max(rms)
                contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
                contrast_mean = np.mean(contrast)
                tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
                audio_features.append({
                    "rms_mean": rms_mean,
                    "rms_max": rms_max,
                    "tempo": tempo,
                    "contrast_mean": contrast_mean
                })
                excitement_labels.append(item["excitement_level"])
        high_excitement_indices = [i for i, label in enumerate(excitement_labels) if label > 0.7]
        if high_excitement_indices:
            high_excitement_features = [audio_features[i] for i in high_excitement_indices]
            self.excitement_thresholds = {
                "rms_mean": np.mean([f["rms_mean"] for f in high_excitement_features]),
                "rms_max": np.mean([f["rms_max"] for f in high_excitement_features]),
                "tempo": np.mean([f["tempo"] for f in high_excitement_features]),
                "contrast_mean": np.mean([f["contrast_mean"] for f in high_excitement_features])
            }
        thresholds_path = os.path.join(self.model_dir, "excitement_thresholds.json")
        with open(thresholds_path, 'w') as f:
            json.dump(self.excitement_thresholds, f)
        print(f"Excitement thresholds calibrated and saved to {thresholds_path}")
        return self.excitement_thresholds

    def analyze_performance(self, test_queries):
        print("Analyzing performance on test queries...")
        results = []
        frames, timestamps, audio_path, frame_features, scenes = self.preprocess_video()
        for query in test_queries:
            start_time = time.time()
            summary = self.generate_summary(query, frames, timestamps, audio_path, frame_features, scenes)
            end_time = time.time()
            processing_time = end_time - start_time
            results.append({
                "query": query,
                "summary_duration": summary["duration"],
                "num_scenes": len(summary["scenes"]),
                "processing_time": processing_time
            })
        avg_processing_time = np.mean([r["processing_time"] for r in results])
        avg_summary_duration = np.mean([r["summary_duration"] for r in results])
        avg_scenes = np.mean([r["num_scenes"] for r in results])
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.bar(range(len(results)), [r["processing_time"] for r in results])
        plt.xticks(range(len(results)), [f"Q{i+1}" for i in range(len(results))])
        plt.xlabel("Query")
        plt.ylabel("Processing Time (s)")
        plt.title("Query Processing Times")
        plt.subplot(1, 2, 2)
        plt.bar(range(len(results)), [r["summary_duration"] for r in results])
        plt.xticks(range(len(results)), [f"Q{i+1}" for i in range(len(results))])
        plt.xlabel("Query")
        plt.ylabel("Summary Duration (s)")
        plt.title("Summary Durations")
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "performance_analysis.png"))
        print(f"Average processing time: {avg_processing_time:.2f}s")
        print(f"Average summary duration: {avg_summary_duration:.2f}s")
        print(f"Average number of scenes: {avg_scenes:.2f}")
        return {
            "results": results,
            "avg_processing_time": avg_processing_time,
            "avg_summary_duration": avg_summary_duration,
            "avg_scenes": avg_scenes
        }

    def save_model(self, path=None):
        if path is None:
            path = os.path.join(self.model_dir, "summarizer_model")
        os.makedirs(path, exist_ok=True)
        with open(os.path.join(path, "attention_weights.json"), 'w') as f:
            json.dump(self.attention_weights, f)
        with open(os.path.join(path, "excitement_thresholds.json"), 'w') as f:
            json.dump(self.excitement_thresholds, f)
        if hasattr(self, 'attention_network_trained') and self.attention_network_trained:
            torch.save(self.attention_network.state_dict(), os.path.join(path, "attention_network.pt"))
        self.yolo_model.save(os.path.join(path, "yolo_model.pt"))
        print(f"Model saved to {path}")
        return path

    def load_model(self, path):
        print(f"Loading model from {path}")
        weights_path = os.path.join(path, "attention_weights.json")
        if os.path.exists(weights_path):
            with open(weights_path, 'r') as f:
                self.attention_weights = json.load(f)
        thresholds_path = os.path.join(path, "excitement_thresholds.json")
        if os.path.exists(thresholds_path):
            with open(thresholds_path, 'r') as f:
                self.excitement_thresholds = json.load(f)
        network_path = os.path.join(path, "attention_network.pt")
        if os.path.exists(network_path):
            self.attention_network.load_state_dict(torch.load(network_path))
            self.attention_network_trained = True
        yolo_path = os.path.join(path, "yolo_model.pt")
        if os.path.exists(yolo_path):
            self.yolo_model = YOLO(yolo_path)
        print("Model loaded successfully")
        return True

In [ ]:
def main(video_path, query, output_dir=os.path.join(BASE_DIR, "output"), dataset_path=None):

    summarizer = SportsVideoSummarizer(
        video_path=video_path,
        output_dir=output_dir,
        frame_rate= 1
    )

    if dataset_path:
        print("Fine-tuning models...")
        summarizer.finetune_object_detection(dataset_path)
        summarizer.optimize_attention_weights(dataset_path)
        summarizer.train_attention_network(dataset_path)
        summarizer.calibrate_excitement_detection(dataset_path)

    print("Processing video...")
    frames, timestamps, audio_path, frame_features, scenes = summarizer.preprocess_video()

    print(f"Generating summary for query: {query}")
    summary = summarizer.generate_summary(
        query, frames, timestamps, audio_path, frame_features, scenes
    )

    print(f"Summary video created: {summary['summary_video_path']}")
    print(f"Summary duration: {summary['duration']:.2f} seconds")
    print(f"Number of scenes: {len(summary['scenes'])}")

    summarizer.save_model()

In [ ]:
if __name__ == "__main__":

    video_path = "/content/2932301-uhd_4096_2160_24fps.mp4"
    output_dir = os.path.join(BASE_DIR, "output")
    dataset_path = None

    query = "football"

    main(video_path, query, output_dir, dataset_path)

Loading models...


  warnings.warn(

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
  self.db_client.recreate_collection(



Initialization complete.
Processing video...
Preprocessing video: /content/2932301-uhd_4096_2160_24fps.mp4
Video properties: FPS=24.00, Duration=15.30s, Estimated Frames=367
Total frames extracted: 16

16

Audio already extracted: /content/drive/MyDrive/sports_video_summarizer/output/2932301-uhd_4096_2160_24fps_audio.wav
Processing frame 1/16 at 0.00s

0: 352x640 7 persons, 2 cars, 1 bench, 175.4ms
Speed: 4.5ms preprocess, 175.4ms inference, 1.9ms postprocess per image at shape (1, 3, 352, 640)
Processing frame 2/16 at 1.00s

0: 352x640 8 persons, 2 cars, 2 benchs, 1 sports ball, 124.7ms
Speed: 3.5ms preprocess, 124.7ms inference, 1.2ms postprocess per image at shape (1, 3, 352, 640)
Processing frame 3/16 at 2.00s

0: 352x640 9 persons, 2 cars, 1 sports ball, 129.6ms
Speed: 4.0ms preprocess, 129.6ms inference, 1.6ms postprocess per image at shape (1, 3, 352, 640)
Processing frame 4/16 at 3.00s

0: 352x640 7 persons, 1 car, 2 sports balls, 126.3ms
Speed: 3.6ms preprocess, 126.3ms infere

MoviePy - Done.
Moviepy - Writing video /content/drive/MyDrive/sports_video_summarizer/output/summaries/summary_20260106_045753.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/sports_video_summarizer/output/summaries/summary_20260106_045753.mp4
Summary video saved to /content/drive/MyDrive/sports_video_summarizer/output/summaries/summary_20260106_045753.mp4
Summary video created: /content/drive/MyDrive/sports_video_summarizer/output/summaries/summary_20260106_045753.mp4
Summary duration: 2.00 seconds
Number of scenes: 1
Model saved to /content/drive/MyDrive/sports_video_summarizer/output/models/summarizer_model
